# SignScribe: English (ASL) letter models

Trains, evaluates and exports the two files the web app loads for English letters:
`letters-mlp.json` and `letters-knn.json` (ASL keeps these original names, see `src/recognition/models.ts`).
24 static letters (J and Z move: they need `$1` templates from webcam recordings, not pictures). It runs the repo's own scripts
at commit `c88ee4e`, so features and file format match the app. **This model has never been trained before.**

**Dataset:** Images of American Sign Language (ASL) Alphabet Gestures (Sowrow, Iftekhar, Prova, Huq; East West University), CC BY 4.0,
https://data.mendeley.com/datasets/48dg9vhmyk/2. 210,000 pictures, 28 classes (A–Z, Del, Space), 3 signers.
Del and Space are not letters of the app's ASL alphabet: they are counted and skipped.

**Nobody on the team has opened this dataset yet**, so step 3 *discovers* the folder layout and which part of the path is the
signer, prints it, and stops if it is unsure. Look at what it prints before going on.

**Kaggle settings:** Internet **On**; Accelerator **None**. Attach the `Root.zip` you already downloaded as a **private** Kaggle
dataset (*+ Add Input*), or let the notebook try Mendeley's public API.

**Slow step:** MediaPipe on every picture. `MAX_PER_CLASS_PER_SIGNER` keeps a random subset (fixed seed) so a first run is quick;
raise it (or set None for all 210,000) once the pipeline works.

## Before you share anything from this notebook
- **Keep this notebook and its output private.** The `*-knn.json` file stores real hand landmark samples
  (the repo never commits it), and the takes file holds landmarks of every picture.
- **Only quote numbers that a script wrote** (`MODEL_RESULTS.md`, the `reports/` files). These are numbers on **dataset pictures**,
  not on a webcam: real-camera accuracy is still **NOT MEASURED**.
- `MODEL_RESULTS.md` in the output is the repo's file with this run appended by `ml/evaluate.py`; it replaces `docs/MODEL_RESULTS.md`
  in a branch (then update section 6 of `docs/PROJECT_STATUS.md`; `npm run check` compares the two).


## 1. Settings

In [ ]:
REPO_URL = "https://github.com/Youssef-Adel91/SignScribe.git"
REPO_COMMIT = "c88ee4e"      # main on 2026-11-01; set to None for the newest main
NOTEBOOK_NAME = "signscribe_asl_letters"

INPUT_PATH = None                  # e.g. "/kaggle/input/asl-alphabet-gestures"; None = whatever is attached, else download
MENDELEY_ID, MENDELEY_VERSION = "48dg9vhmyk", 2
DATASET_NAME = "ASL Alphabet Gestures (East West University)"
DATASET_VERSION = "Mendeley 48dg9vhmyk v2, as published"

MAX_PER_CLASS_PER_SIGNER = 500     # pictures per letter per signer (random, seed 0); None = all
SEED = 0
SIGNER_DEPTH = None                # None = find it; or the index of the folder level that names the signer
SIGNER_REGEX = None                # or a regex on the file name with group 1 = signer (used when no folder level fits)
SIGNING_HAND = "right"             # unknown for this dataset; mirror augmentation covers the other hand

REIMPORT = False
KEEP_TAKES = False

## 2. Setup

In [ ]:
# ---- setup: paths, a helper that runs a command and streams its output ----
import hashlib, json, os, random, re, shutil, subprocess, sys, time, urllib.request, zipfile
from pathlib import Path

IN_KAGGLE = Path("/kaggle/working").exists()
WORK = Path("/kaggle/working") if IN_KAGGLE else Path.cwd() / "work"
SCRATCH = Path("/kaggle/temp") if Path("/kaggle/temp").exists() else Path("/tmp/signscribe")
WORK.mkdir(parents=True, exist_ok=True)
SCRATCH.mkdir(parents=True, exist_ok=True)

# only for checking this notebook itself with generated hands: never set it for a real run
PIPELINE_TEST = os.environ.get("SIGNSCRIBE_PIPELINE_TEST") == "1"
if PIPELINE_TEST:
    print("PIPELINE TEST MODE: generated data, the results mean nothing")

def run(cmd, cwd=None):
    """Runs a command, prints its output while it runs, stops the notebook if it fails."""
    cmd = [str(c) for c in cmd]
    print("$", " ".join(cmd), flush=True)
    p = subprocess.Popen(cmd, cwd=cwd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True,
                         env={**os.environ, "PYTHONUNBUFFERED": "1"})
    for line in p.stdout:
        print(line, end="")
    if p.wait() != 0:
        raise RuntimeError(f"command failed with exit code {p.returncode}: {' '.join(cmd[:3])} ...")

def sha256_of(path, n=16):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1 << 20), b""):
            h.update(chunk)
    return h.hexdigest()[:n]

print("work folder (kept as notebook output):", WORK)
print("scratch folder (not kept):", SCRATCH)

In [ ]:
# ---- get the SignScribe code (the same ml/ scripts the team runs) + MediaPipe + the hand model ----
REPO = Path(os.environ.get("SIGNSCRIBE_REPO_DIR", SCRATCH / "SignScribe"))
if not (REPO / ".git").exists():
    run(["git", "clone", "--quiet", REPO_URL, REPO])
if REPO_COMMIT and not os.environ.get("SIGNSCRIBE_REPO_DIR"):
    run(["git", "checkout", "--quiet", REPO_COMMIT], cwd=REPO)
COMMIT = subprocess.check_output(["git", "rev-parse", "--short", "HEAD"], cwd=REPO, text=True).strip()
print("SignScribe commit:", COMMIT)

try:
    import mediapipe  # noqa: F401
except ImportError:
    run([sys.executable, "-m", "pip", "install", "--quiet", "mediapipe>=0.10.14,<0.11", "numpy"])
import mediapipe
print("mediapipe", mediapipe.__version__)

# the same file and URL as tools/get-models.mjs (npm run models)
HAND_MODEL = REPO / "public" / "models" / "hand_landmarker.task"
HAND_MODEL_URL = "https://storage.googleapis.com/mediapipe-models/hand_landmarker/hand_landmarker/float16/1/hand_landmarker.task"
if not HAND_MODEL.exists() and not PIPELINE_TEST:
    HAND_MODEL.parent.mkdir(parents=True, exist_ok=True)
    urllib.request.urlretrieve(HAND_MODEL_URL, HAND_MODEL)
if HAND_MODEL.exists():
    print("hand model:", HAND_MODEL.name, f"{HAND_MODEL.stat().st_size/1e6:.1f} MB, sha256", sha256_of(HAND_MODEL))

In [ ]:
# ---- dataset helpers: use an attached Kaggle dataset, else download from Mendeley Data ----
IMAGE_SUFFIXES = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}

def extract_all_zips(folder):
    """Extracts every .zip under `folder` next to itself (zips inside zips too), then deletes the extracted zip copy."""
    done = set()
    while True:
        zips = [z for z in Path(folder).rglob("*.zip") if z not in done]
        if not zips:
            return
        for z in zips:
            target = z.with_suffix("")
            print("extracting", z.relative_to(folder), flush=True)
            with zipfile.ZipFile(z) as zf:
                zf.extractall(target)
            done.add(z)
            z.unlink()  # only copies in the scratch folder are ever deleted

def mendeley_download(dataset_id, version, dest):
    """Downloads every file of a public Mendeley Data dataset through its public API."""
    dest = Path(dest)
    dest.mkdir(parents=True, exist_ok=True)
    def get_json(url):
        req = urllib.request.Request(url, headers={"Accept": "application/json", "User-Agent": "signscribe-notebook"})
        with urllib.request.urlopen(req, timeout=60) as r:
            return json.load(r)
    base = f"https://data.mendeley.com/public-api/datasets/{dataset_id}"
    folder_ids = ["root"]
    try:
        folder_ids += [f["id"] for f in get_json(f"{base}/folders/{version}")]
    except Exception as e:
        print("(no folder list:", e, ")")
    got = 0
    for fid in folder_ids:
        for f in get_json(f"{base}/files?folder_id={fid}&version={version}"):
            url = (f.get("content_details") or {}).get("download_url")
            if not url:
                continue
            out = dest / f["filename"]
            size = (f.get("content_details") or {}).get("size") or f.get("size")
            print(f"downloading {f['filename']} ({(size or 0)/1e6:.0f} MB)", flush=True)
            urllib.request.urlretrieve(url, out)
            got += 1
    if not got:
        raise RuntimeError("the Mendeley API listed no files")

def get_dataset(dest, input_path, dataset_id, version, page):
    """Returns a scratch folder with the dataset's files, zips extracted. Never writes into /kaggle/input."""
    dest = Path(dest)
    marker = dest / ".ready"
    if marker.exists():
        print("dataset already prepared in", dest)
        return dest
    dest.mkdir(parents=True, exist_ok=True)
    src = Path(input_path) if input_path else None
    if src is None and Path("/kaggle/input").exists() and any(Path("/kaggle/input").iterdir()):
        src = Path("/kaggle/input")
    if src is not None:
        print("using the attached files in", src)
        for z in src.rglob("*.zip"):  # zips are copied so they can be extracted; pictures are linked, not copied
            rel = z.relative_to(src)
            (dest / rel).parent.mkdir(parents=True, exist_ok=True)
            shutil.copy2(z, dest / rel)
        link = dest / "attached"
        if not link.exists():
            link.symlink_to(src.resolve(), target_is_directory=True)
    else:
        try:
            mendeley_download(dataset_id, version, dest)
        except Exception as e:
            raise RuntimeError(
                f"Automatic download failed ({e}).\n"
                f"Download the dataset in your browser from {page} , upload the zip as a PRIVATE Kaggle dataset, "
                f"attach it with '+ Add Input' (right panel) and run this cell again. Or set INPUT_PATH."
            ) from e
    extract_all_zips(dest)
    marker.write_text("ok")
    return dest

def images_under(folder):
    """Every picture under `folder`, following the link to an attached dataset."""
    found = []
    for d, _, files in os.walk(folder, followlinks=True):
        found += [Path(d) / f for f in files if Path(f).suffix.lower() in IMAGE_SUFFIXES]
    return sorted(found)

## 3. Get the dataset, find its layout (class folder, signer), take a subset

In [ ]:
RAW = get_dataset(SCRATCH / "asl_raw", INPUT_PATH, MENDELEY_ID, MENDELEY_VERSION,
                  "https://data.mendeley.com/datasets/48dg9vhmyk/2")
sys.path.insert(0, str(REPO / "ml"))
from signscribe_ml.alphabet import load_alphabet
ALPHA = load_alphabet("asl")
DATASET_CLASSES = {chr(c) for c in range(ord("a"), ord("z") + 1)} | {"del", "space", "nothing"}

pics = images_under(RAW)
print("pictures found:", len(pics))
rows = []  # (relative folder parts, file)
for p in pics:
    parts = p.relative_to(RAW).parts[:-1]
    cls_idx = max((i for i, x in enumerate(parts) if x.strip().lower() in DATASET_CLASSES), default=None)
    rows.append((parts, cls_idx, p))
no_class = sum(1 for r in rows if r[1] is None)
print("pictures with no class folder (A..Z, Del, Space) in their path:", no_class)
rows = [r for r in rows if r[1] is not None]
assert rows, "no class folders found: look at the layout below and adapt this cell"

depth_values = {}
for parts, ci, _ in rows:
    for d, x in enumerate(parts):
        if d != ci and x.strip().lower() not in DATASET_CLASSES:
            depth_values.setdefault(d, set()).add(x)
print("folder levels other than the class folder (level: number of different names, examples):")
for d, v in sorted(depth_values.items()):
    print(f"  {d}: {len(v)}  {sorted(v)[:6]}")

def signer_of(parts, p):
    if SIGNER_DEPTH is not None:
        return parts[SIGNER_DEPTH] if SIGNER_DEPTH < len(parts) else None
    if SIGNER_REGEX:
        m = re.search(SIGNER_REGEX, p.name)
        return m.group(1) if m else None
    return None

if SIGNER_DEPTH is None and not SIGNER_REGEX:
    fits = [d for d, v in depth_values.items() if 2 <= len(v) <= 10]
    three = [d for d in fits if len(depth_values[d]) == 3]
    pick = three or fits
    if len(pick) != 1:
        raise RuntimeError("Could not tell which folder level is the signer (see the list above). "
                           "Set SIGNER_DEPTH to that level, or SIGNER_REGEX for a file-name pattern, then run this cell again.")
    SIGNER_DEPTH = pick[0]
    print(f"signer = folder level {SIGNER_DEPTH}: {sorted(depth_values[SIGNER_DEPTH])}  <- check this is right")

by = {}
skipped = {}
for parts, ci, p in rows:
    label = ALPHA.resolve(parts[ci])
    if label is None or ALPHA.kind_of(label) != "static":
        skipped[parts[ci]] = skipped.get(parts[ci], 0) + 1
        continue
    s = signer_of(parts, p)
    if s is None:
        skipped["(no signer)"] = skipped.get("(no signer)", 0) + 1
        continue
    by.setdefault((s, label), []).append(p)
signers = sorted({s for s, _ in by})
SIGNER_ID = {s: f"s{i+1}" for i, s in enumerate(signers)}
print("signers:", {k: v for k, v in SIGNER_ID.items()})
print("not imported (not a static letter of the app's ASL alphabet, or no signer):", skipped)
assert len(signers) >= 2, "leave-one-signer-out needs at least 2 signers"

# a flat folder of links named <signer>_<LETTER>_<n>.<ext>; the import reads letter and signer from the name
rng = random.Random(SEED)
STAGE = SCRATCH / "asl_stage"
if STAGE.exists():
    shutil.rmtree(STAGE)
STAGE.mkdir()
count = {}
for (s, label), files in sorted(by.items()):
    files = sorted(files)
    if MAX_PER_CLASS_PER_SIGNER and len(files) > MAX_PER_CLASS_PER_SIGNER:
        files = sorted(rng.sample(files, MAX_PER_CLASS_PER_SIGNER))
    for n, f in enumerate(files):
        (STAGE / f"{SIGNER_ID[s]}_{label}_{n}{f.suffix.lower()}").symlink_to(f.resolve())
    count[label] = count.get(label, 0) + len(files)
print(f"staged {sum(count.values())} pictures, {len(count)} letters (expected 24):")
print("  ".join(f"{k}:{v}" for k, v in sorted(count.items())))

## 4. Pictures → hand landmarks (MediaPipe) → takes file

In [ ]:
TAKES = SCRATCH / "asl.jsonl"
if REIMPORT or not TAKES.exists():
    run([sys.executable, REPO / "ml/import_images.py", "--images", STAGE, "--out", TAKES, "--overwrite",
         "--alphabet", "asl", "--dataset-name", DATASET_NAME, "--source", "asl-ewu", "--hand", SIGNING_HAND,
         "--label-regex", r"^s\d+_([A-Z])_\d+\.\w+$", "--person-regex", r"^(s\d+)_",
         "--model", HAND_MODEL], cwd=REPO)
else:
    print("using the existing takes file", TAKES, "(set REIMPORT = True to import again)")
print("takes:", sum(1 for _ in open(TAKES)), " sha256", sha256_of(TAKES))

## 5. Honest evaluation: leave-one-signer-out
With only 3 signers, each fold trains on 2 people: expect a lower and more variable number than with many people.

In [ ]:
OUT = WORK / "signscribe-asl-output"
REPORTS, MODELS = OUT / "reports", OUT / "models"
for d in (REPORTS, MODELS):
    d.mkdir(parents=True, exist_ok=True)
test_flag = ["--allow-synthetic"] if PIPELINE_TEST else []
results_md = [] if PIPELINE_TEST else ["--results-md", REPO / "docs/MODEL_RESULTS.md"]
version = DATASET_VERSION + (f", subset of {MAX_PER_CLASS_PER_SIGNER} per letter per signer (seed {SEED})" if MAX_PER_CLASS_PER_SIGNER else "")
before = set(REPORTS.glob("*.json"))
run([sys.executable, REPO / "ml/evaluate.py", "--data", TAKES, "--alphabet", "asl",
     "--dataset-name", DATASET_NAME, "--dataset-version", version,
     "--augment", "all", "--augment-ablation", "--out-dir", REPORTS, *results_md, *test_flag], cwd=REPO)
EVAL_JSON = max(set(REPORTS.glob("*.json")) - before, key=lambda p: p.stat().st_mtime)
print("evaluation report:", EVAL_JSON.with_suffix(".md").name)

## 6. Confidence thresholds (a table to choose from by hand)

In [ ]:
run([sys.executable, REPO / "ml/calibrate.py", "--data", TAKES, "--alphabet", "asl",
     "--dataset-name", DATASET_NAME, "--dataset-version", version, "--out-dir", REPORTS, *test_flag], cwd=REPO)

## 7. Export `letters-mlp.json` and `letters-knn.json` (trained on all signers) and validate them

In [ ]:
common = ["--alphabet", "asl", "--augment", "all", "--seed", "0", *test_flag]
run([sys.executable, REPO / "ml/export_mlp.py", "--data", TAKES, "--out", MODELS / "letters-mlp.json",
     "--evaluation-json", EVAL_JSON, *common], cwd=REPO)
run([sys.executable, REPO / "ml/train_knn.py", "--data", TAKES, "--out", MODELS / "letters-knn.json",
     "--max-per-class", "300", *common], cwd=REPO)
labels = []
for f in (MODELS / "letters-mlp.json", MODELS / "letters-knn.json"):
    run([sys.executable, REPO / "ml/validate_model.py", f, "--alphabet", "asl"], cwd=REPO)
    labels.append(json.loads(f.read_text())["labels"])
assert labels[0] == labels[1], "the two files do not list the same letters in the same order"
print("OK: both files list the same", len(labels[0]), "letters:", " ".join(labels[0]))

## 8. Package the output

In [ ]:
# ---- collect everything the team needs into one folder and one zip (the notebook's output) ----
for p in [REPO / "docs" / "MODEL_RESULTS.md", TAKES.with_suffix(".import.md")]:
    if p.exists():
        shutil.copy2(p, OUT / ("MODEL_RESULTS.md" if p.name == "MODEL_RESULTS.md" else "reports/" + p.name))
if KEEP_TAKES and TAKES.exists():
    shutil.copy2(TAKES, OUT / TAKES.name)
info = {
    "notebook": NOTEBOOK_NAME,
    "signscribe_commit": COMMIT,
    "mediapipe": mediapipe.__version__,
    "hand_model_sha256_16": sha256_of(HAND_MODEL) if HAND_MODEL.exists() else None,
    "dataset": DATASET_NAME, "dataset_version": DATASET_VERSION,
    "takes_sha256_16": sha256_of(TAKES),
    "finished_utc": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()),
    "pipeline_test": PIPELINE_TEST,
}
(OUT / "RUN_INFO.json").write_text(json.dumps(info, indent=1))
archive = shutil.make_archive(str(WORK / OUT.name), "zip", OUT)
print("\nOutput folder:", OUT)
for p in sorted(OUT.rglob("*")):
    if p.is_file():
        print(f"  {p.relative_to(OUT)}  ({p.stat().st_size/1e3:.0f} kB)")
print("zip:", archive)

## 9. Putting the models into the app (on your computer, in a branch)
1. Download the output zip. Copy `models/letters-mlp.json` and `models/letters-knn.json` into `public/models/`.
   **Never commit the k-NN file.** Whether to commit the MLP file is a team decision.
2. Record the run: replace `docs/MODEL_RESULTS.md` with the zip's copy, update `docs/PROJECT_STATUS.md` section 6,
   `docs/CREDITS.md` (this dataset's citation and licence), run `npm run check`, open a PR.
3. Thresholds by hand in `src/recognition/tuning.ts` from `reports/*calibration*`, with a test.
4. Still missing for English: J and Z (`tools/dollarpy/record_templates.py` on a webcam), and the real-webcam test.